


#### NHANES (National Health and Nutrition Examination Survey) in US
**[NHANES 2017-2018 cycle](https://wwwn.cdc.gov/nchs/nhanes/continuousnhanes/default.aspx?BeginYear=2017)**
<br>
Each file is linked by SEQN (Participant ID)
<br>
**About Files:**
<br>
* [Demographics/"DEMO_J.XPT"](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/DEMO_J.htm) = age/sex
* [Osteoporosis/"OSQ_J.XPT"](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/OSQ_J.htm) = fracture history
* [Physical Functioning/"PFQ_J.XPT"](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/PFQ_J.htm) = mobility difficulty
<br>



# NHANES Falls-Risk Cohort Study

**Objective** <br>
To evaluate whether older adults (aged 50+) with self-reported mobility difficulties experience a significantly higher rate of fall-caused bone fractures compared to a matched control group.

* **Data Integration:** Merging multiple NHANES 2017-2018 data files using a shared unique identifier (`SEQN`).
* **Feature Engineering:** Deriving explicit risk-factor (mobility difficulty) and clinical outcome (fall-caused bone fractures) variables from raw survey question responses.
* **Cohort Design:** Constructing a fair comparison cohort using **exact matching** on
  two key confounding demographic variables  **Age** (within ±2 years) and **Sex**.
  I chose exact matching over Propensity Score Matching (PSM) here because I'm only
  controlling for two variables; PSM's main advantage is balancing many covariates
  simultaneously via a modeled score, which adds real value once more confounders
  (e.g., income, comorbidities) are incorporated. With a small matched sample, PSM
  can also produce unstable scores. A natural next step for this project  or for
  the real Falls Clinic study, which likely has more confounders and larger
  administrative datasets would be to move to PSM.
* **Statistical Analysis:** Executing robust statistical testing to evaluate differences in fracture rates between the matched groups.<br>

This project serves as a direct pipeline practice run for the methodologies required in the **Falls Clinic** role:
* **Real-World Translation:** The pipeline directly mirrors the workflow of comparing real-world falls-clinic patient outcomes against a matched administrative-data control group.
* **Operational Readiness:** Demonstrates end-to-end data proficiency - from handling raw, complex survey structures to delivering rigorous, defensible clinical insights.


### Research References & Methodological Justification

#### Methodological Anchor: NHANES Cohort Insights
**Reference:** *[PMC9642892 (NHANES, ages 60-85, co-occurring fall-risk factors)](https://pmc.ncbi.nlm.nih.gov/articles/PMC9642892/)*

* **Methodological Validation:** Because NHANES lacks a single, direct "did you fall" questionnaire metric, this paper validates using proxy self-reported functional and health variables as legitimate, published fall-risk indicators. 
* **Key Literature Finding:** The authors found that fall risk factors heavily cluster; older adults rarely present with an isolated risk factor. On average, individuals possess **4 or more co-occurring risk factors**, grouping them into distinct risk profiles (e.g., sensory/cognitive deficits vs. complex multimorbidity).
* **Project Application & Limitations:** This directly justifies using self-reported physical metrics as a proxy for risk. Crucially, the paper's findings provide a vital caveat for the *Limitations* section of this notebook: while this project isolates a single risk factor ("mobility difficulty"), the literature demonstrates that in real-world clinical settings, these risks accumulate multiplicatively.


#### Predictor Validation: The Weight of Mobility Metrics
**Reference:** *[PMC12645780 (NHATS, 94-variable fall-predictor model)](https://pmc.ncbi.nlm.nih.gov/articles/PMC12645780/)*

* **Methodological Validation:** Provides a high-powered, rigorous benchmark proving that functional mobility impairments are among the absolute strongest predictors of fall incidents.
* **Key Literature Finding:** In a massive machine-learning analysis evaluating **94 distinct variables**, functional indicators like lower extremity weakness, poor balance, and difficulty with activities of daily living (ADLs) emerged as paramount warning signs outperforming standard demographic predictors like chronological age in several models.
* **Project Application:** This empirical evidence validates the feature selection strategy for this project. Choosing "mobility difficulty" (extracted from the raw `PFQ_J` questionnaire data) as our primary exposure variable is a highly defensible, evidence-backed choice rather than an arbitrary one.

In [16]:
import pandas as pd
demo = pd.read_sas("DEMO_J.XPT")
osq = pd.read_sas("OSQ_J.XPT")
pfq = pd.read_sas("PFQ_J.XPT")

merged = demo.merge(osq, on="SEQN").merge(pfq, on="SEQN")

In [ ]:
#initial data assessment 

#rows and columns
demoshape = demo.shape
osqshape = osq.shape
pfqshape=pfq.shape
print("Rows and Columns in each datasets:")
print(demoshape)
print(osqshape)
print(pfqshape)



Rows and Columns in each datasets:
(9254, 46)
(3069, 95)
(8421, 36)


In [ ]:
#first 5 rows of each files
#check missing values
print("Demographics dataset")
print(demo.head())
demo.isna().sum()


Demographics dataset
      SEQN  SDDSRVYR  RIDSTATR  RIAGENDR  RIDAGEYR  RIDAGEMN  RIDRETH1  \
0  93703.0      10.0       2.0       2.0       2.0       NaN       5.0   
1  93704.0      10.0       2.0       1.0       2.0       NaN       3.0   
2  93705.0      10.0       2.0       2.0      66.0       NaN       4.0   
3  93706.0      10.0       2.0       1.0      18.0       NaN       5.0   
4  93707.0      10.0       2.0       1.0      13.0       NaN       5.0   

   RIDRETH3  RIDEXMON  RIDEXAGM  ...  DMDHREDZ  DMDHRMAZ  DMDHSEDZ  \
0       6.0       2.0      27.0  ...       3.0       1.0       3.0   
1       3.0       1.0      33.0  ...       3.0       1.0       2.0   
2       4.0       2.0       NaN  ...       1.0       2.0       NaN   
3       6.0       2.0     222.0  ...       3.0       1.0       2.0   
4       7.0       2.0     158.0  ...       2.0       1.0       3.0   

       WTINT2YR      WTMEC2YR  SDMVPSU  SDMVSTRA  INDHHIN2  INDFMIN2  INDFMPIR  
0   9246.491865   8539.731348   

SEQN           0
SDDSRVYR       0
RIDSTATR       0
RIAGENDR       0
RIDAGEYR       0
RIDAGEMN    8657
RIDRETH1       0
RIDRETH3       0
RIDEXMON     550
RIDEXAGM    5821
DMQMILIZ    3250
DMQADFC     8693
DMDBORN4       0
DMDCITZN       3
DMDYRSUS    7306
DMDEDUC3    6948
DMDEDUC2    3685
DMDMARTL    3685
RIDEXPRG    8144
SIALANG        0
SIAPROXY       0
SIAINTRP       0
FIALANG      474
FIAPROXY     474
FIAINTRP     474
MIALANG     2570
MIAPROXY    2570
MIAINTRP    2570
AIALANGA    4277
DMDHHSIZ       0
DMDFMSIZ       0
DMDHHSZA       0
DMDHHSZB       0
DMDHHSZE       0
DMDHRGND       0
DMDHRAGZ       0
DMDHREDZ     490
DMDHRMAZ     191
DMDHSEDZ    4503
WTINT2YR       0
WTMEC2YR       0
SDMVPSU        0
SDMVSTRA       0
INDHHIN2     491
INDFMIN2     474
INDFMPIR    1231
dtype: int64

In [30]:
print("Osteoporosis dataset")
print(osq.head())
osq.isna().sum()


Osteoporosis dataset
      SEQN  OSQ010A  OSQ010B  OSQ010C  OSQ020A  OSQ020B  OSQ020C  OSD030AA  \
0  93705.0      2.0      2.0      2.0      NaN      NaN      NaN       NaN   
1  93708.0      2.0      2.0      2.0      NaN      NaN      NaN       NaN   
2  93709.0      2.0      2.0      2.0      NaN      NaN      NaN       NaN   
3  93711.0      2.0      2.0      2.0      NaN      NaN      NaN       NaN   
4  93713.0      2.0      2.0      2.0      NaN      NaN      NaN       NaN   

   OSQ040AA  OSD050AA  ...  OSQ140U  OSQ150  OSQ160A  OSQ160B  OSQ170  OSQ180  \
0       NaN       NaN  ...      NaN     2.0      NaN      NaN     2.0     NaN   
1       NaN       NaN  ...      NaN     9.0      NaN      NaN     2.0     NaN   
2       NaN       NaN  ...      NaN     2.0      NaN      NaN     2.0     NaN   
3       NaN       NaN  ...      NaN     2.0      NaN      NaN     2.0     NaN   
4       NaN       NaN  ...      NaN     2.0      NaN      NaN     1.0    70.0   

   OSQ190  OSQ200  OSQ2

SEQN          0
OSQ010A       1
OSQ010B       1
OSQ010C       1
OSQ020A    2992
           ... 
OSQ180     2816
OSQ190     3061
OSQ200        1
OSQ210     2984
OSQ220     3061
Length: 95, dtype: int64

In [31]:
print("Physical Functioning Dataset")
print(pfq.head())
pfq.isna().sum()

Physical Functioning Dataset
      SEQN  PFQ020  PFQ030  PFQ033  PFQ041  PFQ049  PFQ051  PFQ054  PFQ057  \
0  93705.0     NaN     NaN     NaN     NaN     2.0     2.0     2.0     2.0   
1  93706.0     2.0     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
2  93707.0     2.0     NaN     NaN     1.0     NaN     NaN     NaN     NaN   
3  93708.0     NaN     NaN     NaN     NaN     1.0     1.0     2.0     1.0   
4  93709.0     NaN     NaN     NaN     NaN     1.0     1.0     1.0     2.0   

   PFQ059  ...  PFQ061Q  PFQ061R  PFQ061S  PFQ061T  PFQ063A  PFQ063B  PFQ063C  \
0     2.0  ...      1.0      1.0      1.0      1.0      NaN      NaN      NaN   
1     NaN  ...      NaN      NaN      NaN      NaN      NaN      NaN      NaN   
2     NaN  ...      NaN      NaN      NaN      NaN      NaN      NaN      NaN   
3     NaN  ...      1.0      1.0      1.0      1.0     10.0     11.0     20.0   
4     NaN  ...      2.0      4.0      1.0      2.0     10.0     14.0     20.0   

   PFQ063D  PFQ

SEQN          0
PFQ020     5571
PFQ030     8313
PFQ033     8339
PFQ041     5857
PFQ049     2852
PFQ051     2852
PFQ054     2852
PFQ057     2852
PFQ059     4571
PFQ061A    5494
PFQ061B    6279
PFQ061C    6279
PFQ061D    5494
PFQ061E    5494
PFQ061F    5494
PFQ061G    5494
PFQ061H    5494
PFQ061I    5494
PFQ061J    5494
PFQ061K    5494
PFQ061L    5494
PFQ061M    5494
PFQ061N    5494
PFQ061O    5494
PFQ061P    5494
PFQ061Q    5494
PFQ061R    5494
PFQ061S    5494
PFQ061T    5494
PFQ063A    6152
PFQ063B    6923
PFQ063C    7346
PFQ063D    7648
PFQ063E    7905
PFQ090     2852
dtype: int64

A large amount of data is missing for two different reasons:
<br>
--> **Random/true missingness**: someone should have answered but didn't (refused/forgot) a real data quality issue.
<br>
--> **Structural missingness**: the question didn't apply to that person, so NHANES's built-in skip logic skipped it automatically which is not an error, and doesn't need fixing.

Most of the high-missingness columns in Osteoporosis and Physical Functioning fall into the second category e.g., fracture-cause questions are only asked of people who already said "yes" to having that type of fracture. This is expected by survey design.

**Demographics**: SEQN (Respondent ID), RIAGENDR (Gender), RIDAGEYR (Age in years) = 0% missing. my two matching variables are complete, so no cleaning is needed for the matching step.

In [33]:
merged.shape

(3069, 175)

## Cohort Assembly

This step defines who is in my study: adults aged 50+ who have both
Osteoporosis and Physical Functioning data available. This required
merging DEMO_J, OSQ_J, and PFQ_J (the merging/mechanical part of cohort
assembly) and filtering to the eligible age range (the criteria-definition
part).

After merging all three files on SEQN, the result has 3,069 rows, matching
the Osteoporosis file exactly. This confirms an inner join: since OSQ_J was
only administered to adults 50+, it's the limiting file, and every one of
those participants also had matching Demographics and Physical Functioning
records. Column count (175) confirms no data was accidentally duplicated:
46 + 95 + 36 = 177 total columns, minus 2 duplicate SEQN columns removed
during the two merge steps = 175.

In [ ]:
#check the merged file has 50+ age range
merged['RIDAGEYR'].describe() 

count    3069.000000
mean       65.398827
std         9.313423
min        50.000000
25%        58.000000
50%        64.000000
75%        73.000000
max        80.000000
Name: RIDAGEYR, dtype: float64

My independent variable is MOBILITY_LIMITED, my dependent variable is FALL_FRACTURE, and I control for the covariates age and sex through matching, since both are known confounders of the fall–mobility relationship

## Defining the Outcome Variable: Design Decision

OSQ_J records up to 5 fracture occurrences per bone site (1st through 5th).
This project uses only the **1st-occurrence** fracture-cause variables
(OSD050AA, OSD050BA, OSD050CA) to define FALL_FRACTURE, since the vast
majority of participants with any fracture reported only one occurrence
(per OSQ020a/b/c frequency counts in the codebook). This is a deliberate
simplification, a small number of participants whose fall-caused
fracture happened on a 2nd, 3rd, 4th, or 5th occurrence will not be
captured. Given how rare repeat fractures are in this sample, this
simplification has minimal impact on the analysis while keeping the
variable construction transparent and easy to audit.

In [45]:
# Confirm missing OSD050AA corresponds to "never fractured" (OSQ010A = 2)
check_hip = merged[merged['OSD050AA'].isna()]
print("OSQ010A values where OSD050AA is missing:")
print(check_hip['OSQ010A'].value_counts())

OSQ010A values where OSD050AA is missing:
OSQ010A
2.0    2989
1.0      26
9.0       2
Name: count, dtype: int64


In [47]:
check_wrist=merged[merged['OSD050BA'].isna()]
print("OSQ010B values where OSD050BA is missing:")
print(check_wrist['OSQ010B'].value_counts())

OSQ010B values where OSD050BA is missing:
OSQ010B
2.0    2750
1.0     211
9.0       3
Name: count, dtype: int64


In [48]:
check_spine =merged[merged['OSD050CA'].isna()]
print("OSQ010C values where OSD050CA is missing:")
print(check_spine['OSQ010C'].value_counts())

OSQ010C values where OSD050CA is missing:
OSQ010C
2.0    2948
1.0      75
9.0       4
Name: count, dtype: int64


## Checking My Own Assumption

I said above that fracture-cause questions are "only asked of people who
already said yes" — so a missing value should always mean "no fracture."
I tested this instead of just assuming it.

**What I checked:** for each bone site, when the "cause of fracture" answer
is missing, does the "did you ever fracture this?" answer always say "No"?

**What I found:**

| Bone  | Missing "cause" AND said "No fracture" | Missing "cause" BUT said "Yes, fractured" |
|-------|------------------------------------------|---------------------------------------------|
| Hip   | 2,989                                     | **26**                                       |
| Wrist | 2,750                                     | **211**                                      |
| Spine | 2,948                                      | **75**                                       |

**In simple words:** most of the time, my assumption was right — missing
really did mean "never fractured." But **312 people total** said "yes,
I did fracture this bone," and yet the reason for that fracture is
blank. This isn't because the question skipped them — NHANES cannot ask
about a 2nd fracture without a 1st one already being recorded, so these
people did answer the 1st-occurrence question. The specific *cause*
answer for that 1st fracture is simply missing — most likely because
they refused to answer, said "don't know," or there's a small gap in
how that answer was recorded.

**What I'm doing about it:** I can't recover the true cause for these
312 people, so I'm keeping my straightforward rule (missing cause =
not counted as a fall) and being upfront that this makes my
fall-fracture count a slight *undercount*. These 312 people are
fracture cases I genuinely can't classify as fall-caused or not, and
they default to "not a fall" in my analysis — which isn't fully
accurate for them specifically, but is the most honest way to handle
an answer I don't have.

In [ ]:
'''
Outcome variable: did this person fracture a bone as a result of a fall?
Using 1st-occurrence cause variables only

--> OSD050aa - Reason hip fracture occurred 1st time
--> OSD050ba - Reason wrist fracture occurred 1st time
--> OSD050ca - Reason spine fracture occurred 1st time
'''

fall_hip = merged['OSD050AA'].isin([1, 2])
fall_wrist = merged['OSD050BA'].isin([1, 2])
fall_spine = merged['OSD050CA'].isin([1, 2])

merged['FALL_FRACTURE'] = (fall_hip | fall_wrist | fall_spine).astype(int)
# 0 = fall fracture at all and 1 = fall fracture
print(merged['FALL_FRACTURE'].value_counts())

FALL_FRACTURE
0    2942
1     127
Name: count, dtype: int64


In [52]:
# Count how many people have fall-fracture at 2+ sites
overlap = (fall_hip.astype(int) + fall_wrist.astype(int) + fall_spine.astype(int))
print(overlap.value_counts())

0    2942
1     110
2      16
3       1
Name: count, dtype: int64


## Verifying the FALL_FRACTURE Count

Individually, fall-caused fractures (codes 1 or 2) totaled:
- Hip: 37 (25 standing-height + 12 hard fall)
- Wrist: 85 (58 standing-height + 27 hard fall)
- Spine: 23 (12 standing-height + 11 hard fall)
- Sum: 145 fracture *events*

However, FALL_FRACTURE = 1 for only 127 unique *people*, not 145. This
18-person gap is explained by the OR logic in my code: a person with
fall-caused fractures at more than one site (e.g., both hip and wrist)
is counted once in my final variable, but contributes to two of the
individual site totals above. This confirms my combination logic is
working correctly, it's counting people, not fracture events, and
the overlap (18 people with multiple fall-fracture sites) is a
plausible, expected pattern rather than a data error.

I confirmed this directly by counting how many bone sites each person
was affected at:

| Sites affected | People |
|---|---|
| 0 | 2,942 |
| 1 | 110 |
| 2 | 16 |
| 3 | 1 |

At least 1 site: 110 + 16 + 1 = **127**  matches FALL_FRACTURE=1 exactly.
Total site-hits: 110 + (16×2) + (1×3) = **145**  matches the individual
site sum exactly. Both checks confirm the variable is correctly built.

In [ ]:
'''
Predictor variable: does this person have mobility difficulty?

--> PFQ054 - Need special equipment to walk
--> PFQ061B - Difficulty walking for a quarter mile
--> PFQ061C - Difficulty walking up ten stairs
--> PFQ061I - Difficulty standingup from armless chair
'''


needs_equipment = merged['PFQ054'] == 1
walk_difficulty = merged['PFQ061B'].isin([2, 3, 4])
stairs_difficulty = merged['PFQ061C'].isin([2, 3, 4])
chair_difficulty = merged['PFQ061I'].isin([2, 3, 4])

merged['MOBILITY_LIMITED'] = (
    needs_equipment | walk_difficulty | stairs_difficulty | chair_difficulty
).astype(int)
# 0 = no mobility limitation at all and 1 = mobility limitation 
print(merged['MOBILITY_LIMITED'].value_counts())

MOBILITY_LIMITED
0    1726
1    1343
Name: count, dtype: int64


## Why These 3 PFQ Variables Specifically

PFQ_J contains 20 difficulty variables (PFQ061A-T) covering a wide range
of activities. I selected only PFQ061B (walking distance), PFQ061C
(climbing stairs), and PFQ061I (standing from a chair) because these
specifically capture balance, gait, and lower-body strength, the
physical mechanisms most directly linked to falling. This mirrors the
NHATS fall-predictor study (Kohler-Voinov et al.), where chair-stand,
balance, and walking scores were among the strongest independent
predictors of falls. Other PFQ variables (e.g., difficulty using
utensils, managing money) reflect dexterity or cognitive/social burden
rather than fall-specific physical risk, so were excluded to keep the
variable conceptually focused and defensible.

In [53]:
# Check overlap directly
overlap = (
    (merged['PFQ054'] == 1).astype(int) +
    merged['PFQ061B'].isin([2,3,4]).astype(int) +
    merged['PFQ061C'].isin([2,3,4]).astype(int) +
    merged['PFQ061I'].isin([2,3,4]).astype(int)
)
print(overlap.value_counts().sort_index())

0    1726
1     507
2     709
3     127
Name: count, dtype: int64


## Verifying the MOBILITY_LIMITED Count

Individually, the four mobility-difficulty conditions (within my filtered
50+ cohort, not the full PFQ_J sample) each flagged some number of
people. Because MOBILITY_LIMITED combines four conditions with OR logic,
and these conditions overlap heavily (people with poor balance/strength
tend to struggle with several of these tasks at once, not just one),
a large gap is expected between the sum of individual conditions and the
final unique-person count.

I confirmed this directly by counting how many of the four conditions
each person triggered:

| Conditions triggered | People |
|---|---|
| 0 | 1,726 |
| 1 | 507 |
| 2 | 709 |
| 3 | 127 |
| 4 | 0 |

At least 1 condition: 507 + 709 + 127 + 0 = **1,343**  matches
MOBILITY_LIMITED=1 exactly.

**Note on overlap pattern:** unlike FALL_FRACTURE, where 87% of affected
people (110 of 127) had only 1 site involved, here the majority of
affected people (709 of 1,343, ~53%) triggered 2 conditions
simultaneously, and nobody triggered all 4. This reflects a real
difference in the underlying data: fracture sites are fairly independent
events, while the four mobility measures capture overlapping symptoms of
the same physical decline (weak legs, poor balance), so co-occurrence
is naturally much more common here.

In [ ]:
'''
Match each mobility-limited person to a similar person (same sex, similar age)
Who is NOT mobility-limited, so the two groups are fairly comparable

**Matching approach:** for each mobility-limited person, I find one
non-limited person of the same sex (RIAGENDR) and within 2 years of age
(RIDAGEYR), using simple nearest-neighbour exact matching. I chose exact
matching over Propensity Score Matching (PSM) because I'm only
controlling for two variables here -- PSM's advantage comes from
balancing many covariates simultaneously via a modeled score, which
would matter more with a larger set of confounders and a bigger sample.

'''
print(merged.groupby('MOBILITY_LIMITED')['RIDAGEYR'].describe())
print(merged.groupby('MOBILITY_LIMITED')['RIAGENDR'].value_counts())


                   count       mean       std   min   25%   50%   75%   max
MOBILITY_LIMITED                                                           
0                 1726.0  63.078795  8.680990  50.0  56.0  62.0  69.0  80.0
1                 1343.0  68.380491  9.252726  50.0  61.0  68.0  78.0  80.0
MOBILITY_LIMITED  RIAGENDR
0                 1.0         892
                  2.0         834
1                 2.0         715
                  1.0         628
Name: count, dtype: int64


## Pre-Matching Group Comparison

Before matching, the two groups differ meaningfully on both confounders:

**Age:** Mobility-limited participants are on average 5.3 years older
(mean 68.4) than non-limited participants (mean 63.1). This confirms
age is a real confounder in this data, older participants are more
likely to report mobility limitation, and age independently affects
fracture risk, so this difference must be controlled for before
comparing fracture rates between groups.

**Sex:** The mobility-limited group is somewhat more female (53.2%)
than the non-limited group (48.3%). Since fracture risk (particularly
osteoporotic fractures) is known to differ by sex, this is a second
confounder worth correcting for through matching.

These differences justify the matched-cohort approach: without matching,
any observed difference in FALL_FRACTURE rates between groups could
simply reflect these underlying age/sex differences rather than a true
association with mobility limitation.

In [55]:
#Building the matched cohort
def build_matched_cohort(df, age_window=2):
    group_a = df[df['MOBILITY_LIMITED'] == 1].copy()
    group_b_pool = df[df['MOBILITY_LIMITED'] == 0].copy()

    matched_pairs = []
    used_b_indices = set()

    for idx, a_row in group_a.iterrows():
        candidates = group_b_pool[
            (group_b_pool['RIAGENDR'] == a_row['RIAGENDR']) &
            (abs(group_b_pool['RIDAGEYR'] - a_row['RIDAGEYR']) <= age_window) &
            (~group_b_pool.index.isin(used_b_indices))
        ]
        if len(candidates) == 0:
            continue
        candidates = candidates.copy()
        candidates['age_diff'] = abs(candidates['RIDAGEYR'] - a_row['RIDAGEYR'])
        best_match = candidates.sort_values('age_diff').iloc[0]
        used_b_indices.add(best_match.name)
        matched_pairs.append((a_row['SEQN'], best_match['SEQN']))

    print(f"Matched {len(matched_pairs)} pairs out of {len(group_a)} "
          f"mobility-limited participants "
          f"({len(matched_pairs)/len(group_a)*100:.1f}% match rate)")

    matched_ids = [p[0] for p in matched_pairs] + [p[1] for p in matched_pairs]
    return df[df['SEQN'].isin(matched_ids)].copy()

matched_cohort = build_matched_cohort(merged)

Matched 1115 pairs out of 1343 mobility-limited participants (83.0% match rate)


## Matching Method and Results

**Method:** 1:1 nearest-neighbour matching without replacement, using
exact matching on sex (RIAGENDR) and caliper matching (±2 years) on age
(RIDAGEYR). For each mobility-limited participant, I searched the pool
of non-limited participants for candidates of the same sex within 2
years of age, then selected the closest age match. Once selected, a
non-limited participant was removed from the pool (matching without
replacement) to prevent the same person being reused as a match for
multiple mobility-limited participants.

**Result:** 1,115 of 1,343 mobility-limited participants (83.0%) were
successfully matched. The remaining 228 unmatched participants (17.0%)
were likely concentrated at older ages, where fewer non-limited
candidates were available within the matching window, since mobility
limitation becomes more common with age, the pool of eligible "similar
but unaffected" comparisons shrinks as age increases.


In [56]:
unmatched_ids = set(merged[merged['MOBILITY_LIMITED']==1]['SEQN']) - set(matched_cohort[matched_cohort['MOBILITY_LIMITED']==1]['SEQN'])
unmatched = merged[merged['SEQN'].isin(unmatched_ids)]
print(unmatched['RIDAGEYR'].describe())

count    228.000000
mean      79.271930
std        1.828724
min       69.000000
25%       80.000000
50%       80.000000
75%       80.000000
max       80.000000
Name: RIDAGEYR, dtype: float64


## Verifying the Unmatched Cases

I checked the age distribution of the 228 unmatched mobility-limited
participants directly, rather than assuming where the gap came from:

- Mean age: 79.3 (vs. 68.4 for the overall mobility-limited group)
- Median, 25th, and 75th percentiles: all exactly 80
- Minimum age: 69

This confirms the unmatched cases are heavily concentrated at the oldest
end of the age range. Two factors explain this: (1) NHANES top-codes all
ages 80 and older as exactly "80" for privacy, which compresses many
distinct ages into a single value and makes exact/caliper matching
harder at that boundary; and (2) mobility limitation becomes more
prevalent with age, so the pool of non-limited candidates to match
against shrinks precisely where demand for matches is highest.

**Implication:** the matched cohort slightly underrepresents the very
oldest, most severely affected participants. This is a limitation worth
noting -- the matched analysis may understate the true association if
risk is especially elevated in this oldest subgroup, since they are
disproportionately excluded from the matched comparison.

## Table 1 and Statistical Comparison

With the matched cohort assembled (1,115 pairs, 2,230 people), I can now
build a standard "Table 1" summarizing both groups, then run a chi-square
test to check whether the difference in FALL_FRACTURE rates between
mobility-limited and non-limited participants is statistically significant.

In [57]:
def build_table1(df):
    rows = []
    for group_val, group_name in [(1, "Mobility-Limited"), (0, "No Limitation")]:
        subset = df[df['MOBILITY_LIMITED'] == group_val]
        rows.append({
            "Group": group_name,
            "N": len(subset),
            "Mean Age": round(subset['RIDAGEYR'].mean(), 1),
            "% Female": round((subset['RIAGENDR'] == 2).mean() * 100, 1),
            "% Fall-Caused Fracture": round(subset['FALL_FRACTURE'].mean() * 100, 1),
        })
    return pd.DataFrame(rows)

table1_matched = build_table1(matched_cohort)
print(table1_matched)

              Group     N  Mean Age  % Female  % Fall-Caused Fracture
0  Mobility-Limited  1115      66.2      49.1                     5.6
1     No Limitation  1115      66.1      49.1                     2.6


In [58]:
table1_unmatched = build_table1(merged)
print(table1_unmatched)

              Group     N  Mean Age  % Female  % Fall-Caused Fracture
0  Mobility-Limited  1343      68.4      53.2                     6.9
1     No Limitation  1726      63.1      48.3                     2.0


## Table 1 Results: Before and After Matching

| | Mean Age Gap | % Female Gap | Fall-Fracture Rate (Limited vs Non-Limited) |
|---|---|---|---|
| Before matching | 5.3 years | 4.9 points | 6.9% vs 2.0% (ratio ≈3.5x) |
| After matching | 0.1 years | 0.0 points | 5.6% vs 2.6% (ratio ≈2.2x) |

Matching successfully balanced age and sex between groups (both gaps
reduced to near-zero). Importantly, the fall-fracture rate gap persisted
after matching, though it narrowed somewhat -- indicating that part of
the raw (unmatched) difference was attributable to age/sex confounding,
but a substantial association between mobility limitation and
fall-caused fracture remains even after controlling for these factors.

In [59]:
from scipy import stats

contingency = pd.crosstab(matched_cohort['MOBILITY_LIMITED'], matched_cohort['FALL_FRACTURE'])
print(contingency)

chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
print(f"\nChi-square statistic: {chi2:.3f}")
print(f"p-value: {p_value:.4f}")

FALL_FRACTURE        0   1
MOBILITY_LIMITED          
0                 1086  29
1                 1053  62

Chi-square statistic: 11.731
p-value: 0.0006


## Results

**Matching successfully balanced the confounders:** mean age (66.2 vs
66.1) and % female (49.1% vs 49.1%) were nearly identical between groups
after matching, compared to a 5.3-year age gap before matching.

**Contingency table (matched cohort):**

|                    | No Fracture | Fall-Fracture |
|--------------------|-------------|---------------|
| No Limitation      | 1,086       | 29            |
| Mobility-Limited    | 1,053       | 62            |

**Fall-fracture rates:** 2.6% (no limitation) vs 5.6% (mobility-limited)

**Chi-square test:** χ² = 11.731, p = 0.0006

The difference in fall-caused fracture rates between mobility-limited
and non-limited participants is statistically significant (p < 0.001),
even after matching removed the confounding influence of age and sex.
Mobility-limited participants had more than double the fall-fracture
rate of non-limited participants (5.6% vs 2.6%).

This finding is consistent with prior literature: both PMC9642892 (NHANES,
ages 60-85) and PMC12645780 (NHATS, 11-year longitudinal analysis)
identified mobility and balance limitations as among the strongest
predictors of fall risk in older adults. This project's cross-sectional,
matched-cohort design cannot establish causation or temporal order (see
Limitations), but it does provide evidence of a robust association
consistent with the broader research base.

## Limitations

This project is a practice exercise using public survey data to rehearse
the research methods needed for the real Falls Clinic study, not a
substitute for it. Several limitations should be kept in mind when
interpreting the results:

**1. Cross-sectional design, not longitudinal.**
DEMO_J, OSQ_J, and PFQ_J are all collected at a single visit per
participant. This allows testing for *association* (do mobility-limited
people currently have a higher rate of having ever had a fall-fracture?)
but not *prediction over time* or causal order (does mobility limitation
precede and cause future falls, or could reduced mobility instead be a
*consequence* of a prior fracture?). A true longitudinal design would
require a dataset like NHATS, which re-interviews the same participants
annually (as used in Kohler-Voinov et al., PMC12645780).

**2. No direct "did you fall" question.**
This NHANES cycle has no single question asking whether a participant
fell in the past year. FALL_FRACTURE, the outcome used here, is a
literature-informed proxy (a fracture specifically caused by a fall,
per PMC9642892's approach) but understates true fall frequency, since
many falls do not result in a fracture.

**3. Outcome variable captures only 1st-occurrence fractures.**
OSQ_J records up to 5 fracture events per bone site, but this project
uses only the 1st-occurrence cause variables (OSD050AA/BA/CA). Verification
showed 312 participants had a confirmed

## Conclusion

**Research question:** Among adults aged 50+, do those with self-reported
mobility limitations have a higher rate of fall-caused bone fractures
than those without, after matching on age and sex?

**Answer:** Yes. After matching 1,115 mobility-limited participants to
1,115 non-limited participants of the same sex and within 2 years of
age -- which successfully balanced both confounders (mean age 66.2 vs
66.1; % female 49.1% vs 49.1%) -- mobility-limited participants had a
significantly higher rate of fall-caused fractures (5.6% vs 2.6%,
χ² = 11.731, p = 0.0006).

**What this project demonstrated:**
- Initial data assessment, including distinguishing structural from
  random missingness and verifying that distinction against the data
  rather than assuming it
- Cohort assembly: merging multiple data files on a shared ID and
  applying eligibility criteria
- Deriving outcome and risk-factor variables from raw survey questions,
  including auditing my own logic (the overlap checks on FALL_FRACTURE
  and MOBILITY_LIMITED) rather than trusting the code blindly
- Building and verifying a matched comparison cohort to control for
  known confounders
- Running and interpreting a statistical test appropriate for a binary
  outcome (chi-square)
- Grounding variable choices and interpretation in existing literature
  (PMC9642892, PMC12645780, PHAC's falls surveillance report)

**What I would do differently with real clinical/administrative data:**
- Use a true longitudinal design to test temporal order, not just
  association
- Incorporate additional confounders (comorbidities, frailty, prior
  falls) via logistic regression or Propensity Score Matching
- Use a direct fall-occurrence outcome rather than a fracture-based proxy
- Examine multiple risk factors jointly, given evidence that fall risk
  factors cluster rather than act independently

This practice project mirrors the core research pipeline required for
evaluating the Calgary Falls Assessment Clinic -- cohort matching against
a comparison group, deriving clinical outcome variables, and testing
whether the observed differences are statistically meaningful -- using
public data as a stand-in for the real administrative dataset.